In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
df4 = pd.read_pickle('../data/df4_with_regions.pkl')
print(f"Loaded df4_with_regions: {df4.shape}")

Loaded df4_with_regions: (7679332, 83)


for visualisation suite) 
- function to call values of current transfer timing distribution by age group and time of day
- function to call distribution of spatial transfer location pairs
- temporal transfer pattern distribution (y axis average transfer time, x axis hour of day) 


In [ ]:
# Cell 3: Verify filter value
print(df4['final_termination_reason_spatial'].value_counts().head(10))

final_termination_reason_spatial
last_stage | null_time_gap                                   2457224
candidate_transfer                                           1599906
return_or_intermediate | exceeds_total_allowance             1529845
exceeds_total_allowance                                      1087435
return_or_intermediate                                        370197
missing_info | null_time_gap | exceeds_distance_threshold     192306
exceeds_total_allowance | exceeds_distance_threshold          174537
null_predicted_walking_time                                    91262
return_or_intermediate | null_predicted_walking_time           65413
missing_info | null_time_gap                                   48520
Name: count, dtype: int64


In [5]:
# Cell 4: Filter to confirmed transfers only (reusable base)
df_transfers = df4[
    df4['final_termination_reason_spatial'] == 'candidate_transfer'
].copy()

df_transfers['age_group'] = df_transfers['PATRON_CATG_DESC_TXT'].map({
    'Student':        '7-19',
    'Adult':          '20-59',
    'Senior Citizen': '60+'
})

df_transfers['hour_of_day'] = df_transfers['next_ENTRY_TM'].dt.hour

print(f"Confirmed transfers: {len(df_transfers):,}")
print(f"Null age_group: {df_transfers['age_group'].isna().sum():,}")
print(f"Null hour_of_day: {df_transfers['hour_of_day'].isna().sum():,}")
print(f"Null time_gap_mins: {df_transfers['time_gap_mins'].isna().sum():,}")

Confirmed transfers: 1,599,906
Null age_group: 0
Null hour_of_day: 0
Null time_gap_mins: 0


In [6]:
# Cell 5: Function 1 — Transfer timing distribution by age group and hour of day
# Output: avg transfer time (time_gap_mins) per age group × hour
# CSV columns: age_group, hour_of_day, avg_transfer_time_mins

def trf_time_distribution_csv(df_trf):
    result = (
        df_trf.groupby(['age_group', 'hour_of_day'])['time_gap_mins']
        .mean()
        .reset_index(name='avg_transfer_time_mins')
        .sort_values(['age_group', 'hour_of_day'])
        .reset_index(drop=True)
    )
    result.to_csv('../data/trf_time_distribution.csv', index=False)
    print(f"Saved trf_time_distribution.csv: {result.shape}")
    return result

def trf_time_distribution(file='../data/trf_time_distribution.csv'):
    return pd.read_csv(file)

In [7]:
# Cell 6: Run Function 1
trf_time_dist_df = trf_time_distribution_csv(df_transfers)
print(trf_time_dist_df)

Saved trf_time_distribution.csv: (57, 3)
   age_group  hour_of_day  avg_transfer_time_mins
0      20-59            5                4.065431
1      20-59            6                3.777964
2      20-59            7                3.945754
3      20-59            8                3.761328
4      20-59            9                4.342368
5      20-59           10                4.745160
6      20-59           11                5.033976
7      20-59           12                5.330169
8      20-59           13                5.698946
9      20-59           14                5.684839
10     20-59           15                5.641158
11     20-59           16                5.509475
12     20-59           17                4.329961
13     20-59           18                4.638108
14     20-59           19                5.028860
15     20-59           20                6.179435
16     20-59           21                6.217855
17     20-59           22                6.239104
18     20

In [8]:
# Cell 7: Function 2 — Spatial transfer location pairs (top-N, long format)
# Output: top N most common origin-destination transfer pairs with counts
# CSV columns: ORIG_STATION_NAME, DEST_STATION_NAME, count

def trf_pair_distribution_csv(df_trf, top_n=50):
    result = (
        df_trf.groupby(['ORIG_STATION_NAME', 'DEST_STATION_NAME'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    result.to_csv('../data/trf_pair_dist.csv', index=False)
    print(f"Saved trf_pair_dist.csv: {result.shape}")
    return result

def trf_pair_distribution(file='../data/trf_pair_dist.csv'):
    return pd.read_csv(file)

In [9]:
# Cell 8: Run Function 2
trf_pair_dist_df = trf_pair_distribution_csv(df_transfers)
print(trf_pair_dist_df.head(10))

Saved trf_pair_dist.csv: (50, 3)
          ORIG_STATION_NAME          DEST_STATION_NAME  count
0          Board W'lands CP                 Kranji Stn   6421
1      Board Johor Bahru CP          Alight W'lands CP   4428
2                    Novena                Newton NSEW   3532
3         Opp ITE Coll Ctrl           Yio Chu Kang Int   3191
4     W'lands Train Checkpt  Woodlands Int Alighting 1   2995
5  Opp Bishan Nth Shop Mall                 Bishan Stn   2657
6              Orchard NSEW                Newton NSEW   2553
7         Bukit Panjang DTL                 Newton DTL   2535
8               Jurong East              Choa Chu Kang   2535
9             Tampines NSEW                  Pasir Ris   2458


In [10]:
# Cell 9: Function 3 — Temporal transfer pattern by hour of day, broken down by patron
# Output: avg transfer time per hour × patron category
# CSV columns: hour_of_day, PATRON_CATG_DESC_TXT, avg_transfer_time_mins

def trf_pattern_distribution_csv(df_trf):
    result = (
        df_trf.groupby(['hour_of_day', 'PATRON_CATG_DESC_TXT'])['time_gap_mins']
        .mean()
        .reset_index(name='avg_transfer_time_mins')
        .sort_values(['hour_of_day', 'PATRON_CATG_DESC_TXT'])
        .reset_index(drop=True)
    )
    result.to_csv('../data/trf_pattern_distribution.csv', index=False)
    print(f"Saved trf_pattern_distribution.csv: {result.shape}")
    return result

def trf_pattern_distribution(file='../data/trf_pattern_distribution.csv'):
    return pd.read_csv(file)

In [11]:
# Cell 10: Run Function 3
trf_pattern_dist_df = trf_pattern_distribution_csv(df_transfers)
print(trf_pattern_dist_df)

Saved trf_pattern_distribution.csv: (57, 3)
    hour_of_day PATRON_CATG_DESC_TXT  avg_transfer_time_mins
0             5                Adult                4.065431
1             5       Senior Citizen                4.506336
2             5              Student                3.061194
3             6                Adult                3.777964
4             6       Senior Citizen                4.668852
5             6              Student                3.627072
6             7                Adult                3.945754
7             7       Senior Citizen                4.540824
8             7              Student                4.073243
9             8                Adult                3.761328
10            8       Senior Citizen                4.554660
11            8              Student                3.782905
12            9                Adult                4.342368
13            9       Senior Citizen                5.695313
14            9              Student     

In [12]:
# Frontend query functions (read from pre-computed CSVs)

_data_dir = '../data'

def get_trf_time_distribution(age_group=None, hour=None):
    """
    Returns avg transfer time distribution by age group and hour of day.

    Inputs:
    - age_group: None (all), or one of '7-19', '20-59', '60+'
    - hour:      None (all), or int 0-23

    Returns: DataFrame with [age_group, hour_of_day, avg_transfer_time_mins]
    """
    df = pd.read_csv(os.path.join(_data_dir, 'trf_time_distribution.csv'))

    if age_group is not None:
        df = df[df['age_group'] == age_group]
    if hour is not None:
        df = df[df['hour_of_day'] == hour]

    return df.reset_index(drop=True)


def get_trf_pair_distribution(orig_station=None, dest_station=None):
    """
    Returns top transfer station pairs by count.

    Inputs:
    - orig_station: None (all), or filter by origin station name string
    - dest_station: None (all), or filter by destination station name string

    Returns: DataFrame with [ORIG_STATION_NAME, DEST_STATION_NAME, count]
    """
    df = pd.read_csv(os.path.join(_data_dir, 'trf_pair_dist.csv'))

    if orig_station is not None:
        df = df[df['ORIG_STATION_NAME'] == orig_station]
    if dest_station is not None:
        df = df[df['DEST_STATION_NAME'] == dest_station]

    return df.reset_index(drop=True)


def get_trf_temporal_pattern(patron=None):
    """
    Returns avg transfer time by hour of day for the temporal pattern chart.

    Inputs:
    - patron: None (all), or one of 'Adult', 'Student', 'Senior Citizen'

    Returns: DataFrame with [hour_of_day, PATRON_CATG_DESC_TXT, avg_transfer_time_mins]
    """
    df = pd.read_csv(os.path.join(_data_dir, 'trf_pattern_distribution.csv'))

    if patron is not None:
        df = df[df['PATRON_CATG_DESC_TXT'] == patron]

    return df.reset_index(drop=True)